In [1]:
import numpy as np
import genesis as gs
import random
from pathlib import Path
import quaternion
from trimesh.transformations import quaternion_from_euler
from scipy.spatial.transform import Rotation as R

[I 04/16/25 16:57:30.497 522201] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:

# init 
gs.init(backend=gs.cuda, precision='32', theme='dark', eps=1e-12)

# create a scene 
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(3, -1, 1.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30, max_FPS=600),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=True,
    show_FPS=False,
)

# entities 
plane = scene.add_entity(gs.morphs.Plane())
bottle_radius = 0.025
bottle_pos = [0.3, 0.0, 0.1] 

plastic_bottle = scene.add_entity(
    gs.morphs.Mesh(
        file="../bottles/plastic_bottle/bottle.stl", 
        pos=bottle_pos,            
        euler=(0, 0, 0.0),
        collision=True,
        visualization=True,
        fixed=True,          
        scale=0.0002                      
    )
)

spot_gripper = scene.add_entity(
    gs.morphs.URDF(
        file='/home/nexus/Desktop/Genesis/genesis/assets/urdf/spot_arm/urdf/open_gripper.urdf',
        euler=(90, 0, 0),
        pos=(-0.2, 0.0, 0.10),
        scale=1.0,
        merge_fixed_links=True,
        fixed = True
    ),
)

# build 
scene.build()

[Genesis] [16:57:33] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [16:57:33] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [16:57:33] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [16:57:33] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.75 GB.
[Genesis] [16:57:34] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [16:57:34] [INFO] Scene <890a863> created.
[Genesis] [16:57:34] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <551ebf0>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [16:57:34] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <3ca3eff>, morph: <gs.morphs.Mesh(file='/home/nexus/Desktop/spot_gen/Exp_RigidEntity/bottles/plastic_bottle/bottle.stl')>, material: <gs.materials.Rigid>.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[Genesis] [16:57:35] [INFO] Adding <gs.RigidEntity>. idx: 2, uid: <5fe01f9>, morph: <gs.morphs.URDF(file='/home/nexus/Desktop/Genesis/genesis/assets/urdf/spot_arm/urdf/open_gripper.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [16:57:35] [INFO] Building scene <890a863>...
[Genesis] [16:57:43] [INFO] Compiling simulation kernels...
[Genesis] [16:58:15] [INFO] Building visualizer...
[Genesis] [16:58:16] [INFO] Viewer created. Resolution: 688×516, max_FPS: 600.


In [3]:

# Define DOFs
hand_dof = np.arange(2)
finger_dof = np.array([1]) 

# Set PD control parameters
spot_gripper.set_dofs_kp(
    np.array([100]*2)
    )
spot_gripper.set_dofs_kv(
    np.array([1]*2)
    )
spot_gripper.set_dofs_force_range(
    np.array([-100]*2),
    np.array([100]*2)
    )
# Grasp parameters
finger_length = 0.09
num_steps = 5
pause_steps = 10
k_dist = 0.3

print("\nGetting the normal with Saved Image data...")


Getting the normal with Saved Image data...


In [4]:
def vector_to_euler(vec):
    vec = np.array(vec) / np.linalg.norm(vec)  # Normalize the vector
    reference = np.array([1, 0, 0])  # Reference vector
    
    if np.allclose(vec, reference):
        return np.array([0.0, 0.0, 0.0])
    
    axis = np.cross(reference, vec)
    angle = np.arccos(np.dot(reference, vec))
    
    if np.linalg.norm(axis) < 1e-6:
        axis = np.array([0, 0, 1])
    else:
        axis = axis / np.linalg.norm(axis)
    
    rotation = R.from_rotvec(angle * axis)
    print(rotation)
    euler_angles = rotation.as_euler('xyz')
    
    return euler_angles

In [5]:
def check_gripper_contacts():
    contacts = spot_gripper.get_contacts()
    print(f"\nRaw contacts data: {contacts}")
    
    if not contacts or 'position' not in contacts:
        print("No contacts detected or invalid contact data :/")
        return False
    
    # Use length of 'position' array for number of contacts
    num_contacts = len(contacts['position'])
    print(f"\033[1;92mNumber of contacts detected: {num_contacts}\033[0m")
    
    # Get link names for mapping
    link_names = [l.name for l in scene.rigid_solver.links]
    print(f"Link names in scene: {link_names}")
    
    # Find indices for bottle and ground
    bottle_link_idx = link_names.index("bottle_stl_baselink") if "bottle_stl_baselink" in link_names else -1
    ground_link_idx = link_names.index("plane_baselink") if "plane_baselink" in link_names else -1
    
    if bottle_link_idx == -1 or ground_link_idx == -1:
        print("Bottle or ground link not found in scene!")
        return False
    
    # Initialize contact flags
    bottle_contact = False
    ground_contact = False
    
    # Gripper finger links for bottle contact
    gripper_finger_links = ["arm_link_wr1", "arm_link_fngr"]
    
    # Process each contact
    for i in range(num_contacts):
        print(f"Contact {i + 1}:")
        print(f"  Position: {contacts['position'][i]}")
        # Infer normal by force direction
        force_b = contacts['force_b'][i]
        norm = np.linalg.norm(force_b)
        normal = force_b / norm if norm > 1e-6 else np.zeros_like(force_b)
        print(f"  Normal (inferred): \033[1;92m{normal}\033[0m")
        print(f"  Force on gripper: \033[1;92m{contacts['force_a'][i]}\033[0m")
        print(f"  Force on other: \033[1;92m{contacts['force_b'][i]}\033[0m")
        link_a_idx = contacts['link_a'][i]
        link_b_idx = contacts['link_b'][i]
        link_a_name = link_names[link_a_idx] if link_a_idx < len(link_names) else 'Unknown'
        link_b_name = link_names[link_b_idx] if link_b_idx < len(link_names) else 'Unknown'
        print(f"  Link A: \033[1;92m{link_a_name}\033[0m")
        print(f"  Link B: \033[1;92m{link_b_name}\033[0m")
        
        # Check for bottle contact (gripper fingers with bottle)
        if link_b_idx == bottle_link_idx and link_a_name in gripper_finger_links:
            print("\033[1;92m  -> Contact with bottle detected!\033[0m")
            bottle_contact = True
        elif link_a_idx == bottle_link_idx and link_b_name in gripper_finger_links:
            print("\033[1;92m  -> Contact with bottle detected!\033[0m")
            bottle_contact = True
        
        # Check for ground contact (any gripper link with ground)
        if (link_b_idx == ground_link_idx and link_a_name.startswith("arm_link_")) or \
           (link_a_idx == ground_link_idx and link_b_name.startswith("arm_link_")):
            print("\033[1;91m  -> Contact with ground detected!\033[0m")
            ground_contact = True
    
    # Apply logic
    if bottle_contact and not ground_contact:
        print("\033[1;92mValid grasp: Gripper fingers contact bottle, no ground contact.\033[0m")
        return True
    elif bottle_contact and ground_contact:
        print("\033[1;91mInvalid grasp: Gripper contacts bottle but also touches ground.\033[0m")
    elif not bottle_contact and ground_contact:
        print("\033[1;91mInvalid grasp: No bottle contact, ground contact detected.\033[0m")
    else:
        print("\033[1;91mInvalid grasp: No bottle contact, no ground contact.\033[0m")
    return False

In [6]:
def pixel_to_world(x, y, depth, cam_pos, cam_lookat, fov, img_width, img_height, world_up):
    """
    Convert pixel coordinates and depth to world coordinates
    
    Args:
        x: pixel x-coordinate (horizontal)
        y: pixel y-coordinate (vertical)
        depth: depth value at that pixel
        cam_pos: camera position (x, y, z)
        cam_lookat: point camera is looking at (x, y, z)
        fov: field of view in degrees
        img_width: width of the image in pixels
        img_height: height of the image in pixels
    
    Returns:
        world_coords: (x, y, z) in world coordinates
    """
    # Convert inputs to numpy arrays
    cam_pos = np.array(cam_pos)
    cam_lookat = np.array(cam_lookat)
    
    # Calculate camera direction vector
    cam_dir = cam_lookat - cam_pos
    cam_dir = cam_dir / np.linalg.norm(cam_dir)
    
    # Calculate camera up vector (assuming world up is (0,0,1))
    world_up = np.array(world_up)
    cam_right = np.cross(cam_dir, world_up)
    cam_right = cam_right / np.linalg.norm(cam_right)
    cam_up = np.cross(cam_right, cam_dir)
    
    # Convert FOV to radians
    fov_rad = np.deg2rad(fov)
    
    # Calculate pixel coordinates in normalized device coordinates (-1 to 1)
    ndc_x = (2.0 * x / (img_width - 1)) - 1.0
    ndc_y = 1.0 - (2.0 * y / (img_height - 1))  # Flip y-axis
    
    # Calculate direction vector in camera space
    aspect_ratio = img_width / img_height
    tan_half_fov = np.tan(fov_rad / 2.0)
    
    cam_space_x = ndc_x * tan_half_fov * aspect_ratio
    cam_space_y = ndc_y * tan_half_fov
    cam_space_z = -1.0  # Negative z is forward in camera space
    
    # Create direction vector in camera space
    cam_space_dir = np.array([cam_space_x, cam_space_y, cam_space_z])
    cam_space_dir = cam_space_dir / np.linalg.norm(cam_space_dir)
    
    # Convert to world space direction
    world_dir = (cam_right * cam_space_dir[0] + 
                cam_up * cam_space_dir[1] + 
                -cam_dir * cam_space_dir[2])
    world_dir = world_dir / np.linalg.norm(world_dir)
    
    # Calculate world position
    world_pos = cam_pos + world_dir * depth
    
    return world_pos

In [7]:
view = {

    "depth": "side_depth.npy",
    "normal_map": "side_normal_map.npy",
    "seg": "side_segmentation_mask.npy",
    "cam_pos": [1, 0, 0.1],
    "up": [0, 0, 1]

}
# Common parameters
cam_lookat = [0.0, 0.0, 0.09]
fov = 30
img_width, img_height = 1280, 960
output_dir = Path("../water_bottle_dataset")
# Load data for the selected view
depth = np.load(output_dir / view["depth"])
normal_map = np.load(output_dir / view["normal_map"])
bottle_seg = np.load(output_dir / view["seg"])
cylinder_pixel_coords = np.where(np.array(bottle_seg) == 1)
cylinder_pixel_coords=np.transpose(cylinder_pixel_coords)
cam_pos = view["cam_pos"]
up = view["up"]

In [13]:
# Select a random cylinder pixel coordinate
random_idx = random.randint(0, len(cylinder_pixel_coords) - 1)
random_y, random_x = cylinder_pixel_coords[random_idx]  
specific_point_depth = depth[random_y, random_x]
specific_point_normal = normal_map[random_y, random_x]
for _ in range(200):
    scene.step()
# Convert normal from [0, 255] to [-1, 1] 
vector_normal = 2 * (specific_point_normal - 127.5) / 255
print(f"Random pixel coordinate from view: ({random_x}, {random_y})")
print(f"Normal vector at random pixel: \033[1;92m{vector_normal}\033[0m")
value = pixel_to_world(random_x, random_y, specific_point_depth, cam_pos, cam_lookat, fov, img_width, img_height, up)
print(f"World coordinates at random pixel: \033[1;92m{value}\033[0m")

print(f"Depth shape: {depth.shape}")
print(f"Normal map shape: {normal_map.shape}")
print(f"Depth at random pixel: {specific_point_depth}")
print(f"Normal vector in Pixels value: \033[1;92m{specific_point_normal}\033[0m")
print(f"Normal vector in Euler: \033[1;92m{vector_to_euler(vector_normal)}\033[0m")

Random pixel coordinate from view: (637, 350)
Normal vector at random pixel: [ 0.99215686 -0.12941176 -0.00392157]
World coordinates at random pixel: [ 0.33240106 -0.00093178  0.14160526]
Depth shape: (960, 1280)
Normal map shape: (960, 1280, 3)
Depth at random pixel: 0.6688947677612305
Normal vector in Pixels value: [254 111 127]
Normal vector in Euler: [-0.00025453  0.00391935 -0.12970254]


In [14]:
quartenion = quaternion_from_euler(* vector_to_euler(-vector_normal))
spot_gripper.set_quat(quartenion)
spot_gripper.set_pos(0.5 * vector_normal+value) 
current_pos = spot_gripper.get_pos()
current_quat = spot_gripper.get_quat()
print(f"Gripper current position & quaternion: \033[1;92m\n{current_pos}\n{current_quat}\033[0m")
for _ in range(200):
    scene.step()

[Genesis] [16:58:46] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [16:58:46] [WARNING] Base link is fixed. Overriding base link pose.


Gripper current position & quaternion: 
tensor([ 0.8285, -0.0656,  0.1396], device='cuda:0')
tensor([ 6.4835e-02, -9.8446e-17, -3.0225e-02,  9.9744e-01], device='cuda:0')


In [15]:
def is_in_xy_plane(quaternion):

    current_quat_array = quaternion.cpu().numpy()
    world_z_axis = np.array([0, 0, 1])
    world_z_quat = quaternion_from_euler(*vector_to_euler(-world_z_axis))
    validation = np.allclose(current_quat_array, world_z_quat, atol=1e-2)
    print(f"Comparing the Gripper with Z: \033[1;92m{validation}\033[0m")
    if validation == False:
        print("\033[1;92mGripper is in XY plane, rotating by pi/2...\033[0m")
        q_rot = quaternion_from_euler(np.pi/2, 0, 0)  
        print(f"Rotation quaternion: \033[1;92m{q_rot}\033[0m")
        gripper_quat = np.quaternion(*current_quat_array) * np.quaternion(*q_rot)
        gripper_quat_array = np.array([gripper_quat.w, gripper_quat.x, gripper_quat.y, gripper_quat.z])
        print(f"New quaternion: \033[1;92m{gripper_quat_array}\033[0m")
        spot_gripper.set_quat(gripper_quat_array)
        spot_gripper.set_pos((k_dist/2) * vector_normal+value)  
        return True     
    else:
        print("\033[1;92mGripper invertical, no need rotation.\033[0m")
        spot_gripper.set_pos((k_dist/2) * vector_normal+value)  

        return False

for i in range(5):
    scene.step()
    
is_in_xy_plane(current_quat)
for _ in range(200):
    scene.step()


/tmp/ipykernel_522201/1150007097.py:27: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  is_in_xy_plane(current_quat)
[Genesis] [16:58:46] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [16:58:46] [WARNING] Base link is fixed. Overriding base link pose.


Comparing the Gripper with Z: False
Gripper is in XY plane, rotating by pi/2...
Rotation quaternion: [0.70710678 0.70710678 0.         0.        ]
New quaternion: [0.04584541 0.04584541 0.68392268 0.72666785]


In [16]:
def check_bottle_contact():
    """
    Check if gripper fingers (arm_link_wr1, arm_link_fngr) contact the bottle (bottle_stl_baselink).
    Returns True if contact is detected, False otherwise.
    """
    contacts = spot_gripper.get_contacts()
    if not contacts or 'position' not in contacts:
        return False
    
    num_contacts = len(contacts['position'])
    link_names = [l.name for l in scene.rigid_solver.links]
    
    bottle_link_idx = link_names.index("bottle_stl_baselink") if "bottle_stl_baselink" in link_names else -1
    if bottle_link_idx == -1:
        print("Bottle link not found in scene!")
        return False
    
    gripper_finger_links = ["arm_link_wr1", "arm_link_fngr"]
    
    for i in range(num_contacts):
        link_a_idx = contacts['link_a'][i]
        link_b_idx = contacts['link_b'][i]
        link_a_name = link_names[link_a_idx] if link_a_idx < len(link_names) else 'Unknown'
        link_b_name = link_names[link_b_idx] if link_b_idx < len(link_names) else 'Unknown'
        
        if link_b_idx == bottle_link_idx and link_a_name in gripper_finger_links:
            print(f"\033[1;92mContact {i+1}: Gripper finger {link_a_name} contacts bottle at {contacts['position'][i]}\033[0m")
            return True
        elif link_a_idx == bottle_link_idx and link_b_name in gripper_finger_links:
            print(f"\033[1;92mContact {i+1}: Gripper finger {link_b_name} contacts bottle at {contacts['position'][i]}\033[0m")
            return True
    
    return False

def perform_grasp(num_steps=10, pause_steps=10, max_gripper_pos=np.pi/2):
    """
    Gradually close the gripper until fingers contact the bottle, then stabilize the grasp.
    Args:
        spot_gripper: Gripper object with get_dofs_position and control_dofs_position methods.
        scene: Genesis scene object for stepping the simulation.
        finger_dof: Index or name of the finger DOF to control.
        num_steps: Number of steps to incrementally close the gripper (max attempts).
        pause_steps: Number of simulation steps per increment for stability.
        max_gripper_pos: Maximum gripper DOF position (radians) to prevent over-closing.
    """

    print("\033[1;92mGrasping...\033[0m")
    current_qpos = spot_gripper.get_dofs_position()
    start_gripper_pos = current_qpos[finger_dof].item()
    gripper_step_size = (max_gripper_pos - start_gripper_pos) / num_steps
    
    print("\033[1;92mExecuting paused grasp...\033[0m")
    for i in range(num_steps + 1):
        current_gripper_pos = start_gripper_pos + i * gripper_step_size
        target_qpos = current_qpos.clone()
        target_qpos[finger_dof] = current_gripper_pos
        spot_gripper.control_dofs_position(target_qpos)
        print(f"Step {i}: Gripper at {current_gripper_pos:.3f}")
        
        # Run simulation steps to stabilize and check contact
        for _ in range(pause_steps):
            scene.step()
        
        # Check for bottle contact
        if check_bottle_contact():
            print("\033[1;92mBottle contact detected, stopping grasp motion.\033[0m")
            break
    
    print("\033[1;92mStabilizing grasp...\033[0m")
    for _ in range(150):
        scene.step()
    
    print(f"\033[1;92mGrasp complete at gripper position: {current_gripper_pos:.3f}\033[0m")
perform_grasp()
for _ in range(200):
    scene.step()

Grasping...
Executing paused grasp...
Step 0: Gripper at 0.345
Contact 1: Gripper finger arm_link_wr1 contacts bottle at [ 0.3        -0.03133192  0.11517504]
Bottle contact detected, stopping grasp motion.
Stabilizing grasp...
Grasp complete at gripper position: 0.345


In [17]:
print(f"\nChecking contacts after grasp... {check_gripper_contacts()}")
for _ in range(200):
    scene.step()



Raw contacts data: {'geom_a': array([1], dtype=int32), 'geom_b': array([5], dtype=int32), 'link_a': array([1], dtype=int32), 'link_b': array([3], dtype=int32), 'position': array([[ 0.29999998, -0.03133202,  0.11517509]], dtype=float32), 'force_a': array([[148.71574 ,  15.822392,  26.269981]], dtype=float32), 'force_b': array([[-148.71574 ,  -15.822392,  -26.269981]], dtype=float32)}
Number of contacts detected: 1
Link names in scene: ['plane_baselink', 'bottle_stl_baselink', 'arm_link_wr0', 'arm_link_wr1', 'arm_link_fngr']
Contact 1:
  Position: [ 0.29999998 -0.03133202  0.11517509]
  Normal (inferred): [-0.97939336 -0.10420111 -0.17300552]
  Force on gripper: [148.71574   15.822392  26.269981]
  Force on other: [-148.71574   -15.822392  -26.269981]
  Link A: bottle_stl_baselink
  Link B: arm_link_wr1
  -> Contact with bottle detected!
Valid grasp: Gripper fingers contact bottle, no ground contact.

Checking contacts after grasp... True
